# NanoBodyBuilder2 structures for the same 6 targets

Rebuilds the six hand-picked targets with NanoBodyBuilder2 and superimposes each on the **same
reference coordinates** the SimpleFold structures were aligned to, so all three (crystal,
SimpleFold, NanoBodyBuilder2) end up in one shared frame and overlay directly.

Framework-only superposition, per-residue error in the B-factor column — identical convention
to the SimpleFold structure run.

In [ ]:
import os, sys, json, glob, time, subprocess, shutil
import numpy as np

WORK, SCRATCH = '/kaggle/working', '/tmp/nbb'
OUT = f'{WORK}/nbb2_structures'; os.makedirs(OUT, exist_ok=True); os.makedirs(SCRATCH, exist_ok=True)

_ds = glob.glob('/kaggle/input/**/targets_vhh.json', recursive=True)
assert len(_ds) == 1, _ds
TG = {f"{t['pdb']}{t['H']}": t for t in json.load(open(_ds[0]))}
PICKS = ['8EMYL', '9G13A', '7B5GF', '7Z1DFFF', '7TPRD', '7FATB']
for p in PICKS:
    assert p in TG, p
    assert 27 - TG[p]['cdr1'][0] < 8, f'{p} would be refused by ImmuneBuilder'
print('all 6 targets present and acceptable to ImmuneBuilder')

def must(c, what):
    p = subprocess.run(c, shell=True, text=True, capture_output=True)
    if p.returncode: raise RuntimeError(f'{what}:\n{(p.stdout+p.stderr)[-1500:]}')

must(f'{sys.executable} -m pip install -q ImmuneBuilder openmm pdbfixer biotite', 'pip')
subprocess.run('apt-get -qq update && apt-get -qq install -y hmmer', shell=True, capture_output=True)
must(f'{sys.executable} -m pip install -q anarci', 'anarci')
import anarci, openmm
from ImmuneBuilder import NanoBodyBuilder2
print('openmm', openmm.__version__)

## Build + superimpose

In [ ]:
import biotite.structure.io.pdb as pdb_io

AA3to1 = {'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G',
          'HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S',
          'THR':'T','TRP':'W','TYR':'Y','VAL':'V'}

def kabsch(P, Q):
    Pm, Qm = P.mean(0), Q.mean(0)
    V, _, W = np.linalg.svd((P - Pm).T @ (Q - Qm))
    D = np.diag([1, 1, np.sign(np.linalg.det(V @ W))])
    return V @ D @ W, Pm, Qm

rmsd = lambda a, b: float(np.sqrt(((a - b) ** 2).sum(-1).mean()))

nb = NanoBodyBuilder2()
report = []
for tid in PICKS:
    t = TG[tid]; Ln = len(t['hseq'])
    raw = f'{SCRATCH}/{tid}_raw.pdb'
    nb.predict({'H': t['hseq']}).save(raw)

    a = pdb_io.PDBFile.read(raw).get_structure(model=1)
    core = a[np.isin(a.atom_name, ['N','CA','C','CB'])]
    seen, order = {}, []
    for ch, ri, ins, rn in zip(core.chain_id, core.res_id, core.ins_code, core.res_name):
        key = (str(ch), int(ri), str(ins))
        if key not in seen:
            seen[key] = len(order); order.append(AA3to1.get(str(rn), 'X'))
    off = t['hseq'].find(''.join(order))
    assert off >= 0, tid

    pred = np.full((Ln, 4, 3), np.nan); ai = {'N':0,'CA':1,'C':2,'CB':3}
    for ch, ri, ins, an, xyz in zip(core.chain_id, core.res_id, core.ins_code, core.atom_name, core.coord):
        pred[seen[(str(ch), int(ri), str(ins))] + off, ai[str(an)]] = xyz
    ref = np.array([[[np.nan]*3 if v is None else v for v in r] for r in t['hca']], float)

    reg = {}
    for k in ('cdr1','cdr2','cdr3'):
        m = np.zeros(Ln, bool); lo, hi = t[k]; m[lo:hi] = True; reg[k] = m
    fw = ~(reg['cdr1'] | reg['cdr2'] | reg['cdr3'])
    ok = np.isfinite(ref[...,0]) & np.isfinite(pred[...,0])

    Rot, Pm, Qm = kabsch(pred[ok & fw[:,None]], ref[ok & fw[:,None]])
    al = np.full_like(pred, np.nan); al[ok] = (pred[ok] - Pm) @ Rot + Qm
    sc = {k: round(rmsd(al[ok & reg[k][:,None]], ref[ok & reg[k][:,None]]), 2) for k in ('cdr1','cdr2','cdr3')}
    sc['framework'] = round(rmsd(al[ok & fw[:,None]], ref[ok & fw[:,None]]), 2)

    d = np.linalg.norm(al - ref, axis=-1)
    dev = np.array([np.nanmean(d[i]) if np.isfinite(d[i]).any() else 0.0 for i in range(Ln)])

    # residue index (0-based, into hseq) for every atom in the FULL structure
    full_idx = {}
    for ch, ri, ins in zip(a.chain_id, a.res_id, a.ins_code):
        k = (str(ch), int(ri), str(ins))
        if k in seen: full_idx[k] = seen[k] + off
    allx = (a.coord - Pm) @ Rot + Qm            # transform every atom, side chains included
    with open(f'{OUT}/{tid}_nbb2.pdb', 'w') as f:
        n = 0
        for ch, ri, ins, an, rn, c in zip(a.chain_id, a.res_id, a.ins_code, a.atom_name,
                                          a.res_name, allx):
            k = (str(ch), int(ri), str(ins))
            if k not in full_idx: continue
            n += 1; b = dev[full_idx[k]]
            f.write(f'ATOM  {n:>5} {str(an):<4}{str(rn):>4} A{full_idx[k]+1:>4}    '
                    f'{c[0]:>8.3f}{c[1]:>8.3f}{c[2]:>8.3f}  1.00{b:>6.2f}          {str(an)[0]:>2}\n')
        f.write('END\n')
    report.append({'tid': tid, 'nbb2_fw_aligned': sc, 'max_res_dev': round(float(dev.max()), 2),
                   'atoms': n, 'seq_offset': int(off)})
    print(f"{tid:<9} cdr3 {sc['cdr3']:>6.2f}  fw {sc['framework']:>5.2f}  ({n} atoms)")

json.dump(report, open(f'{OUT}/nbb2_viz_report.json','w'), indent=1)
print('\nwrote', len(os.listdir(OUT)), 'files')